System Path Setup

In [2]:
# Cell 1: Robust System Path Setup for Project Modules
import os
import sys

notebook_path = os.getcwd() # This should be 'gaias_ark_mangroves/notebooks/'
project_root = os.path.abspath(os.path.join(notebook_path, os.pardir))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root added to sys.path: {project_root}")
print(f"'configs' folder exists at root: {os.path.isdir(os.path.join(project_root, 'configs'))}")
print(f"'regions.py' file exists: {os.path.isfile(os.path.join(project_root, 'configs', 'regions.py'))}")

Project root added to sys.path: c:\Users\pickle-ian\OneDrive\Desktop\Gaia\Gaia's Ark
'configs' folder exists at root: True
'regions.py' file exists: True


Import Core Libraries and GEE Initialization 

In [3]:
# Cell 2: Import Core Libraries and Initialize GEE
import ee
import pandas as pd
import geopandas as gpd
import folium # For interactive mapping
from pygbif import occurrences # For GBIF data
from configs.regions import kenyan_coast_roi # Your ROI

# Initialize GEE (essential for all GEE operations)
ee.Initialize(project='gaias-ark') # <--- REPLACE 'gaias-ark' with YOUR GEE Project ID

print("All core libraries imported and GEE initialized.")

Region of Interest for Kenyan Coast defined.
All core libraries imported and GEE initialized.


Quality Check - Integrated species occurrences data


In [4]:
# Cell 3: Quality Check - Integrated Species Occurrences Data
print("--- Performing Quality Checks on Integrated Species Data ---")

# Load the enriched GBIF data locally
gbif_env_enriched_path = os.path.join(project_root, 'data', 'processed', 'gbif_kenya_mangrove_env_enriched.geojson')
if os.path.exists(gbif_env_enriched_path):
    gbif_local_df = gpd.read_file(gbif_env_enriched_path)
    print(f"Loaded {len(gbif_local_df)} environmentally enriched GBIF records.")
else:
    print(f"Error: Enriched GBIF data not found at {gbif_env_enriched_path}. Please re-run 06_Data_Integration_Species_Mangroves.ipynb.")
    sys.exit("No enriched GBIF data to check.")

if not gbif_local_df.empty:
    # 1. Check for valid geometries (should be points)
    print(f"Number of records with valid geometry: {gbif_local_df.geometry.is_valid.sum()} / {len(gbif_local_df)}")
    if not gbif_local_df.geometry.is_valid.all():
        print("WARNING: Some geometries are invalid. Investigate 'gbif_local_df[~gbif_local_df.geometry.is_valid]'")

    # 2. Check for missing key attributes
    key_cols = ['scientificName', 'decimalLatitude', 'decimalLongitude', 'is_mangrove',
                'elevation_m', 'mean_annual_temp_C', 'total_annual_prec_mm']
    for col in key_cols:
        if col not in gbif_local_df.columns:
            print(f"ERROR: Missing expected column: '{col}'")
        else:
            missing_count = gbif_local_df[col].isnull().sum()
            if missing_count > 0:
                print(f"WARNING: {missing_count} missing values in '{col}'.")

    # 3. Check 'is_mangrove' values (should be 0 or 1)
    if 'is_mangrove' in gbif_local_df.columns:
        unique_is_mangrove = gbif_local_df['is_mangrove'].unique()
        print(f"'is_mangrove' unique values: {unique_is_mangrove}")
        if not all(val in [0, 1] for val in unique_is_mangrove):
            print("ERROR: 'is_mangrove' contains unexpected values (should be 0 or 1).")

    # 4. Check environmental values against expected ranges (sanity check)
    # Note: These are rough checks; actual valid ranges can vary greatly.
    if 'elevation_m' in gbif_local_df.columns:
        if (gbif_local_df['elevation_m'] < -50).any() or (gbif_local_df['elevation_m'] > 200).any(): # Coastal range
            print("WARNING: 'elevation_m' contains values outside typical coastal range (-50 to 200m).")
    
    if 'mean_annual_temp_C' in gbif_local_df.columns:
        if (gbif_local_df['mean_annual_temp_C'] < 10).any() or (gbif_local_df['mean_annual_temp_C'] > 40).any(): # Tropical range
            print("WARNING: 'mean_annual_temp_C' contains values outside typical tropical range (10 to 40C).")

    if 'total_annual_prec_mm' in gbif_local_df.columns:
        if (gbif_local_df['total_annual_prec_mm'] < 100).any() or (gbif_local_df['total_annual_prec_mm'] > 5000).any(): # Tropical range
            print("WARNING: 'total_annual_prec_mm' contains values outside typical tropical range (100 to 5000mm).")

    print("Species data quality checks complete.")
else:
    print("No species data loaded for quality checks.")

--- Performing Quality Checks on Integrated Species Data ---
Loaded 1 environmentally enriched GBIF records.
Number of records with valid geometry: 1 / 1
ERROR: Missing expected column: 'mean_annual_temp_C'
ERROR: Missing expected column: 'total_annual_prec_mm'
'is_mangrove' unique values: [1]
Species data quality checks complete.


Quality Check - Consolidated carbon analysis image

In [5]:
# Cell 4: Quality Check - Consolidated Carbon Analysis Image
print("--- Performing Quality Checks on Consolidated Carbon Image ---")

# Load the consolidated carbon analysis image from GEE Assets
my_gee_project_id = 'gaias-ark'
consolidated_carbon_asset_id = f'projects/{my_gee_project_id}/assets/gaias_ark_carbon_analysis_image_kenya'

# Check if the asset exists (important if the export task failed)
try:
    carbon_analysis_image = ee.Image(consolidated_carbon_asset_id)
    # Attempt to get info to confirm existence
    _ = carbon_analysis_image.getInfo() # Try to force evaluation
    print(f"Loaded consolidated carbon image asset: {consolidated_carbon_asset_id}")
except Exception as e:
    print(f"ERROR: Consolidated carbon image asset not found or inaccessible: {e}")
    print("Please ensure the GEE export task for this asset completed successfully.")
    sys.exit("Cannot proceed with carbon image quality checks.")


# 1. Check band names (should be 'mangrove_presence' and 'AGC_Density_tonnes_C_ha')
expected_bands = ['mangrove_presence', 'AGC_Density_tonnes_C_ha']
actual_bands = carbon_analysis_image.bandNames().getInfo()
print(f"Actual bands in image: {actual_bands}")
if not all(band in actual_bands for band in expected_bands):
    print(f"ERROR: Expected bands {expected_bands} not found in the image. Found: {actual_bands}")

# 2. Sample values within the ROI to check ranges (server-side check)
# We'll create a random sample of points within the ROI and inspect their values.
# Note: This operation can be slow if maxPixels is very high or ROI is large.
sample_points_count = 100 # A small number for quick check

# Ensure there's enough area to sample
if kenyan_coast_roi.area().getInfo() > 1000: # Check if ROI area is meaningful
    sampled_values = carbon_analysis_image.sample(
        region=kenyan_coast_roi,
        scale=30,
        numPixels=sample_points_count,
        seed=0, # For reproducibility
        dropNulls=False # Keep nulls to see if there are gaps
    ).getInfo()['features']
    
    if sampled_values:
        print(f"Sampled {len(sampled_values)} points for quality check.")
        
        # Convert sampled values to a DataFrame for easier inspection
        sample_df = pd.DataFrame([f['properties'] for f in sampled_values])
        
        print("\nSampled Carbon Image Values (head):")
        print(sample_df.head())

        # Check 'mangrove_presence' band
        if 'mangrove_presence' in sample_df.columns:
            unique_presence = sample_df['mangrove_presence'].unique()
            print(f"'mangrove_presence' unique values (sample): {unique_presence}")
            if not all(pd.isna(val) or val in [0, 1] for val in unique_presence):
                print("WARNING: 'mangrove_presence' contains unexpected values (should be 0 or 1).")
        
        # Check 'AGC_Density_tonnes_C_ha' band against expected ranges
        if 'AGC_Density_tonnes_C_ha' in sample_df.columns:
            min_agc = sample_df['AGC_Density_tonnes_C_ha'].min()
            max_agc = sample_df['AGC_Density_tonnes_C_ha'].max()
            print(f"AGC Density range (sample): {min_agc:.2f} to {max_agc:.2f} tonnes C/ha")
            
            # Sanity check: AGC should be non-negative, and typically below a very high threshold
            # Expected max based on your initial vis params of 150.
            if min_agc < 0 or max_agc > 500: # Very generous upper bound for sanity
                print("WARNING: 'AGC_Density_tonnes_C_ha' contains values outside expected range (0 to 500 tonnes C/ha).")
    else:
        print("No samples retrieved from carbon image. ROI might be too small or image is empty.")
else:
    print("ROI area is too small for meaningful sampling.")

print("Carbon image quality checks complete.")

--- Performing Quality Checks on Consolidated Carbon Image ---
Loaded consolidated carbon image asset: projects/gaias-ark/assets/gaias_ark_carbon_analysis_image_kenya
Actual bands in image: ['mangrove_presence', 'AGC_Density_tonnes_C_ha']
Sampled 100 points for quality check.

Sampled Carbon Image Values (head):
   AGC_Density_tonnes_C_ha  mangrove_presence
0                      0.0                  0
1                      0.0                  0
2                      0.0                  0
3                      0.0                  0
4                      0.0                  0
'mangrove_presence' unique values (sample): [0 1]
AGC Density range (sample): 0.00 to 12.52 tonnes C/ha
Carbon image quality checks complete.
